# 🎬 HyperFlow H3 Flash Studio - ComfyUI Multi-GPU Edition
### Dual-T4 Offloading & Experimental DisTorch2 Sharding (`ComfyUI-MultiGPU`)
* **Model:** `VideoRebirthAI/hyperflow-h3-flash` (ref2va_int8_convrot_comfyui)
* **Accelerator:** Kaggle Dual Nvidia Tesla T4 GPU (`GPU T4 x2`)
* **Multi-GPU Strategy:**
  * `CLIPLoaderMultiGPU` & `VAELoaderMultiGPU` pin the Qwen3-VL 32B text encoder and Video/Audio VAEs onto **`cuda:1`**.
  * `HyperFlowLoaderDisTorch2MultiGPU` wraps `HyperFlowLoader` with `ComfyUI-MultiGPU` DisTorch2 (`cuda:0,50%;cuda:1,50%`) to shard the 14B INT8 blocks across both T4s.


In [ ]:
# [Step 1/4] Environment & ComfyUI Multi-GPU Setup
import os, sys, subprocess, time

try:
    sys.stdout.reconfigure(line_buffering=True)
    sys.stderr.reconfigure(line_buffering=True)
except Exception:
    pass

t_total_start = time.time()
t_step1_start = time.time()

print('=' * 75, flush=True)
print('>>> [1/4] Installing ComfyUI, HyperFlow & ComfyUI-MultiGPU Nodes...', flush=True)
print('=' * 75, flush=True)

# Install Cloudflared
subprocess.run('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared', shell=True, check=True)

# Clone ComfyUI core
if not os.path.exists('ComfyUI'):
    subprocess.run('git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git', shell=True, check=True)

# Clone HyperFlow node
custom_nodes_path = 'ComfyUI/custom_nodes/ComfyUI-HyperFlow-Flash'
if not os.path.exists(custom_nodes_path):
    subprocess.run('git clone --depth 1 https://github.com/Video-Rebirth/ComfyUI-HyperFlow-Flash.git ' + custom_nodes_path, shell=True, check=True)

# Clone ComfyUI-MultiGPU (DisTorch2)
multigpu_path = 'ComfyUI/custom_nodes/ComfyUI-MultiGPU'
if not os.path.exists(multigpu_path):
    subprocess.run('git clone --depth 1 https://github.com/pollockjj/ComfyUI-MultiGPU.git ' + multigpu_path, shell=True, check=True)

# Create bridge node (prefixed with zzz_ so ComfyUI loads it after both HyperFlow-Flash and MultiGPU)
bridge_dir = 'ComfyUI/custom_nodes/zzz_ComfyUI-HyperFlow-MultiGPU'
os.makedirs(bridge_dir, exist_ok=True)
bridge_code = '''"""Bridge node wrapping HyperFlowLoader with ComfyUI-MultiGPU DisTorch2."""
import importlib
import logging
import sys
from nodes import NODE_CLASS_MAPPINGS as GLOBAL_MAPPINGS

logger = logging.getLogger('HyperFlowMultiGPU')
NODE_CLASS_MAPPINGS = {}
NODE_DISPLAY_NAME_MAPPINGS = {}

try:
    mgpu_pkg = None
    for name, mod in list(sys.modules.items()):
        if name.endswith('ComfyUI-MultiGPU') or (hasattr(mod, '__file__') and mod.__file__ and 'ComfyUI-MultiGPU' in mod.__file__ and name.count('.') == 0):
            mgpu_pkg = name
            break
    if mgpu_pkg is None:
        mgpu_mod = importlib.import_module('ComfyUI-MultiGPU.wrappers')
    else:
        mgpu_mod = importlib.import_module(f'{mgpu_pkg}.wrappers')
    HyperFlowLoader = GLOBAL_MAPPINGS['HyperFlowLoader']
    NODE_CLASS_MAPPINGS['HyperFlowLoaderDisTorch2MultiGPU'] = mgpu_mod.override_class_with_distorch_safetensor_v2(HyperFlowLoader)
    NODE_DISPLAY_NAME_MAPPINGS['HyperFlowLoaderDisTorch2MultiGPU'] = 'HyperFlowLoader DisTorch2 (MultiGPU)'
    logger.info('[HyperFlowMultiGPU] Registered HyperFlowLoaderDisTorch2MultiGPU')
except Exception as e:
    logger.warning(f'[HyperFlowMultiGPU] Failed to wrap HyperFlowLoader with DisTorch2: {e}')
'''
with open(os.path.join(bridge_dir, '__init__.py'), 'w', encoding='utf-8') as f:
    f.write(bridge_code)

pkgs = ['huggingface_hub>=0.28.0', 'hf_transfer', 'gguf', 'pillow', 'requests', 'tqdm']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'ComfyUI/requirements.txt'], check=True)
t_step1 = time.time() - t_step1_start
print(f'⏱️ [TIMING] Step 1 (ComfyUI & MultiGPU Setup): {t_step1:.2f}s', flush=True)


In [ ]:
# [Step 2/4] High-Speed Model Ingestion (Mounted Dataset + HF Fallback)
import os, shutil, glob, time
from huggingface_hub import hf_hub_download

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
t_step2_start = time.time()

print('=' * 75, flush=True)
print('>>> [2/4] Provisioning Model Checkpoints...', flush=True)
print('=' * 75, flush=True)

os.makedirs('ComfyUI/models/diffusion_models', exist_ok=True)
os.makedirs('ComfyUI/models/text_encoders', exist_ok=True)
os.makedirs('ComfyUI/models/vae', exist_ok=True)

ds_candidates = glob.glob('/kaggle/input/*minimax*weights*')
ds_root = ds_candidates[0] if ds_candidates else '/kaggle/input/minimax-h3-open-weights'
print(f'Detected Mounted Dataset Root: {ds_root} (Exists: {os.path.exists(ds_root)})', flush=True)

def link_or_fetch(rel_ds_path, local_dst, hf_repo, hf_file):
    t_start_item = time.time()
    src = os.path.join(ds_root, rel_ds_path)
    if os.path.exists(src):
        if not os.path.exists(local_dst):
            try:
                os.symlink(src, local_dst)
            except Exception:
                shutil.copyfile(src, local_dst)
        dur = time.time() - t_start_item
        print(f'✓ [MOUNTED 0-SEC] Linked in {dur:.3f}s: {local_dst}', flush=True)
    else:
        if not os.path.exists(local_dst):
            print(f'⏳ [DOWNLOADING FROM HF] {hf_file}...', flush=True)
            p = hf_hub_download(repo_id=hf_repo, filename=hf_file)
            try:
                os.symlink(p, local_dst)
            except Exception:
                shutil.copyfile(p, local_dst)
            dur = time.time() - t_start_item
            size_gb = os.path.getsize(local_dst) / (1024**3)
            print(f'✓ [DOWNLOADED] {local_dst} ({size_gb:.2f} GB) in {dur:.2f}s', flush=True)
        else:
            print(f'✓ [CACHED] Already present: {local_dst}', flush=True)

link_or_fetch('clip/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors', 'ComfyUI/models/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors', 'Comfy-Org/MiniMax-H3', 'text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors')
link_or_fetch('vae/minimax_h3_video_vae_int8_convrot.safetensors', 'ComfyUI/models/vae/minimax_h3_video_vae_fp16.safetensors', 'Comfy-Org/MiniMax-H3', 'vae/minimax_h3_video_vae_fp16.safetensors')
link_or_fetch('vae/minimax_h3_audio_vae_fp32.safetensors', 'ComfyUI/models/vae/minimax_h3_audio_vae_fp32.safetensors', 'Comfy-Org/MiniMax-H3', 'vae/minimax_h3_audio_vae_fp32.safetensors')
link_or_fetch('diffusion_models/minimax_h3_ref2va_hyperflow_flash_14b-int8_convrot.safetensors', 'ComfyUI/models/diffusion_models/minimax_h3_ref2va_hyperflow_flash_14b-int8_convrot.safetensors', 'VideoRebirthAI/hyperflow-h3-flash', 'ref2va_int8_convrot_comfyui/minimax_h3_ref2va_hyperflow_flash_14b-int8_convrot.safetensors')
t_step2 = time.time() - t_step2_start
print(f'⏱️ [TIMING] Step 2 (Total Model Provisioning Time): {t_step2:.2f}s', flush=True)


In [ ]:
# [Step 3/4] Ingest Example References & Pre-load Multi-GPU Patched Workflows
import os, json, shutil, urllib.request

# 1. Input reference images
input_dir = 'ComfyUI/input'
os.makedirs(input_dir, exist_ok=True)

official_urls = {
    'ref_image_1.png': 'https://raw.githubusercontent.com/Video-Rebirth/hyperflow-h3-flash/main/integrations/ComfyUI-HyperFlow-Flash/example_workflows/hyperflow_ref2v_int8.jpg',
    'ref_image_2.png': 'https://raw.githubusercontent.com/Video-Rebirth/hyperflow-h3-flash/main/integrations/ComfyUI-HyperFlow-Flash/example_workflows/hyperflow_i2v_int8.jpg'
}
for fname, url in official_urls.items():
    dst = os.path.join(input_dir, fname)
    if not os.path.exists(dst):
        try:
            urllib.request.urlretrieve(url, dst)
        except Exception as e:
            print(f'Notice on {fname}: {e}')

# 2. Copy official workflows and patch loader nodes for ComfyUI-MultiGPU + DisTorch2
wf_dir = 'ComfyUI/user/default/workflows'
os.makedirs(wf_dir, exist_ok=True)
node_wf = 'ComfyUI/custom_nodes/ComfyUI-HyperFlow-Flash/example_workflows'

def patch_wf_multigpu(wf):
    for node in wf.get('nodes', []):
        ntype = node.get('type')
        if ntype == 'HyperFlowLoader':
            node['type'] = 'HyperFlowLoaderDisTorch2MultiGPU'
            node['title'] = 'HyperFlowLoader DisTorch2 (MultiGPU)'
            vals = list(node.get('widgets_values', ['minimax_h3_fl2va_hyperflow_flash_14b-int8_convrot.safetensors', 'auto']))
            node['widgets_values'] = vals[:2] + ['cuda:0', 7.0, 'cuda:1', 'cuda:0,50%;cuda:1,50%', True]
            if 'widgets_values_named' in node:
                node['widgets_values_named'].update({
                    'device': 'cuda:0',
                    'virtual_vram_gb': 7.0,
                    'donor_device': 'cuda:1',
                    'expert_mode_allocations': 'cuda:0,50%;cuda:1,50%',
                    'eject_models': True,
                })
            node['properties'] = {'Node name for S&R': 'HyperFlowLoaderDisTorch2MultiGPU'}
        elif ntype == 'CLIPLoader':
            node['type'] = 'CLIPLoaderMultiGPU'
            node['title'] = 'CLIPLoader (MultiGPU -> cuda:1)'
            vals = list(node.get('widgets_values', ['qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors', 'minimax', 'default']))
            node['widgets_values'] = vals[:3] + ['cuda:1']
            if 'widgets_values_named' in node:
                node['widgets_values_named']['device'] = 'cuda:1'
            node['properties'] = {'Node name for S&R': 'CLIPLoaderMultiGPU'}
        elif ntype == 'VAELoader':
            node['type'] = 'VAELoaderMultiGPU'
            node['title'] = 'VAELoader (MultiGPU -> cuda:1)'
            vals = list(node.get('widgets_values', []))
            node['widgets_values'] = vals[:1] + ['cuda:1']
            if 'widgets_values_named' in node:
                node['widgets_values_named']['device'] = 'cuda:1'
            node['properties'] = {'Node name for S&R': 'VAELoaderMultiGPU'}
    return wf

if os.path.exists(node_wf):
    for f in os.listdir(node_wf):
        if f.endswith('.json'):
            with open(os.path.join(node_wf, f), 'r', encoding='utf-8') as fp:
                wf_data = patch_wf_multigpu(json.load(fp))
            with open(os.path.join(wf_dir, f), 'w', encoding='utf-8') as fp:
                json.dump(wf_data, fp, indent=2)

print(f'✓ Pre-loaded & Multi-GPU patched {len(os.listdir(wf_dir))} workflows into ComfyUI menu!', flush=True)
print('✓ Reference input assets ready in ComfyUI/input/', flush=True)


In [ ]:
# [Step 4/4] Start ComfyUI Server & Cloudflare Live Web UI
import subprocess, threading, time, re, urllib.request

PORT = 8188
t_step4_start = time.time()
print('=' * 75, flush=True)
print('>>> [4/4] Launching Interactive ComfyUI Server & Cloudflare Tunnel...', flush=True)
print('=' * 75, flush=True)

comfy_proc = subprocess.Popen(f'{sys.executable} ComfyUI/main.py --listen 127.0.0.1 --port {PORT} --enable-cors-header', shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

def monitor():
    for line in comfy_proc.stdout:
        if any(w in line for w in ['Loaded', 'Prompt', 'Executing', 'HyperFlow', 'error', 'Error']):
            print(f'[ComfyUI] {line.strip()}', flush=True)

threading.Thread(target=monitor, daemon=True).start()

tunnel_proc = subprocess.Popen(f'/usr/local/bin/cloudflared tunnel --url http://127.0.0.1:{PORT}', shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

public_url = None
t_tunnel = time.time()
while time.time() - t_tunnel < 40:
    line = tunnel_proc.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if m:
        public_url = m.group(0)
        break

t_step4 = time.time() - t_step4_start
t_total_ready = time.time() - t_total_start

print('\n' + '=' * 80, flush=True)
if public_url:
    print(f'🌐 COMFYUI LIVE WEB UI: {public_url}', flush=True)
    print('👉 Open the link above in your browser to interact with the ComfyUI canvas!', flush=True)
print('=' * 80, flush=True)
print(f'⏱️ [TIMING] Server Launch & Tunnel Ready: {t_step4:.2f}s', flush=True)
print(f'⏱️ [TIMING] 🔥 TOTAL TIME FROM BOOT TO LIVE URL: {t_total_ready:.2f}s ({t_total_ready/60:.2f} min)', flush=True)
print('=' * 80 + '\n', flush=True)

counter = 0
while True:
    time.sleep(60)
    counter += 1
    if counter % 5 == 0:
        print(f'[Heartbeat] ComfyUI interactive server is online ({counter} min).', flush=True)
